[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emersonnjsantos/visao_computacional/blob/main/ATIVIDADE_3/relatorio_atividade_3_projeto.ipynb)


# Atividade 3 — Filtragem de Imagens e Análise de Ruído
## Unidade 2 | Capítulo 1 | Tarefa 3

**Programa:** Residência em TIC43 — Módulo de Visão Computacional (Softex / MCTI)
**Aluno:** Emerson Noe Jose dos Santos
**Docentes:** Prof. Alyson Bezerra | Prof. Matheus Araújo
**Repositório do Projeto:** [github.com/emersonnjsantos/visao_computacional](https://github.com/emersonnjsantos/visao_computacional)

**Objetivo:** Aplicar técnicas fundamentais de filtragem no domínio da frequência, explorando filtros passa-baixa gaussianos, filtros passa-alta, inserção de ruído gaussiano e sal e pimenta, e avaliar o impacto desses processos em imagens do mini-dataset.

In [ ]:
!git clone https://github.com/emersonnjsantos/visao_computacional.git

In [ ]:
# Importações e Configurações

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# Configurações do matplotlib
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

# Caminho base do dataset (Colab após git clone)
DATASET_PATH = '/content/visao_computacional/ATIVIDADE_1/visao_computacional_mini_dataset/dataset'

print("✅ Bibliotecas importadas com sucesso!")
print(f"OpenCV versão: {cv2.__version__}")
print(f"NumPy versão: {np.__version__}")

---
## Parte 1 — Seleção das Imagens

Seleção de uma imagem da **Classe A** e uma da **Classe B** do mini-dataset criado na atividade anterior.

In [ ]:
# Caminhos das imagens selecionadas
path_A = os.path.join(DATASET_PATH, 'classe_A', 'img001_original.jpg')
path_B = os.path.join(DATASET_PATH, 'classe_B', 'img006_original.jpg')

# Carregar imagens
img_A = cv2.imread(path_A)
img_B = cv2.imread(path_B)

# Converter BGR -> RGB para exibição correta no matplotlib
img_A_rgb = cv2.cvtColor(img_A, cv2.COLOR_BGR2RGB)
img_B_rgb = cv2.cvtColor(img_B, cv2.COLOR_BGR2RGB)

# Exibir informações das imagens
print("=" * 60)
print("INFORMAÇÕES DAS IMAGENS SELECIONADAS")
print("=" * 60)
print(f"\n Imagem Classe A: img001_original.jpg")
print(f"   Resolução: {img_A.shape[1]} x {img_A.shape[0]} pixels")
print(f"   Canais: {img_A.shape[2]} (BGR/RGB)")
print(f"   Formato: JPG")
print(f"   Tamanho do arquivo: {os.path.getsize(path_A) / 1024:.1f} KB")

print(f"\n📷 Imagem Classe B: img006_original.jpg")
print(f"   Resolução: {img_B.shape[1]} x {img_B.shape[0]} pixels")
print(f"   Canais: {img_B.shape[2]} (BGR/RGB)")
print(f"   Formato: JPG")
print(f"   Tamanho do arquivo: {os.path.getsize(path_B) / 1024:.1f} KB")

# Exibir imagens originais
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(img_A_rgb)
axes[0].set_title('Imagem Original — Classe A\n(img001_original.jpg)', fontsize=13, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(img_B_rgb)
axes[1].set_title('Imagem Original — Classe B\n(img006_original.jpg)', fontsize=13, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

---
## Parte 2 — Inserção de Ruído

Geração de versões ruidosas de cada imagem utilizando dois tipos de ruído:
1. **Ruído Gaussiano** — simula ruído eletrônico de sensores
2. **Ruído Sal e Pimenta** — simula pixels defeituosos ou erros de transmissão

In [ ]:
def adicionar_ruido_gaussiano(imagem, media=0, desvio_padrao=30):
    """
    Adiciona ruído gaussiano à imagem.

    Parâmetros:
        imagem: imagem de entrada (numpy array)
        media: média da distribuição gaussiana (padrão: 0)
        desvio_padrao: desvio padrão da distribuição (padrão: 30)

    Retorna:
        Imagem com ruído gaussiano aplicado
    """
    ruido = np.random.normal(media, desvio_padrao, imagem.shape).astype(np.float32)
    imagem_ruidosa = cv2.add(imagem.astype(np.float32), ruido)
    imagem_ruidosa = np.clip(imagem_ruidosa, 0, 255).astype(np.uint8)
    return imagem_ruidosa


def adicionar_ruido_sal_pimenta(imagem, probabilidade=0.05):
    """
    Adiciona ruído sal e pimenta à imagem.

    Parâmetros:
        imagem: imagem de entrada (numpy array)
        probabilidade: fração total de pixels afetados (padrão: 0.05 = 5%)

    Retorna:
        Imagem com ruído sal e pimenta aplicado
    """
    imagem_ruidosa = np.copy(imagem)
    total_pixels = imagem.shape[0] * imagem.shape[1]

    # Sal (pixels brancos)
    num_sal = int(total_pixels * probabilidade / 2)
    coords_y = np.random.randint(0, imagem.shape[0], num_sal)
    coords_x = np.random.randint(0, imagem.shape[1], num_sal)
    imagem_ruidosa[coords_y, coords_x] = 255

    # Pimenta (pixels pretos)
    num_pimenta = int(total_pixels * probabilidade / 2)
    coords_y = np.random.randint(0, imagem.shape[0], num_pimenta)
    coords_x = np.random.randint(0, imagem.shape[1], num_pimenta)
    imagem_ruidosa[coords_y, coords_x] = 0

    return imagem_ruidosa

print(" Funções de ruído definidas com sucesso!")

In [ ]:
# Parâmetros do ruído gaussiano
MEDIA = 0
DESVIO_PADRAO = 30

# Aplicar ruído gaussiano nas duas imagens
img_A_gauss = adicionar_ruido_gaussiano(img_A, media=MEDIA, desvio_padrao=DESVIO_PADRAO)
img_B_gauss = adicionar_ruido_gaussiano(img_B, media=MEDIA, desvio_padrao=DESVIO_PADRAO)

print(f"Parâmetros do Ruído Gaussiano: média = {MEDIA}, desvio padrão = {DESVIO_PADRAO}")

# Exibir comparação: Original vs Ruído Gaussiano
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

axes[0, 0].imshow(cv2.cvtColor(img_A, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title('Classe A — Original', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(cv2.cvtColor(img_A_gauss, cv2.COLOR_BGR2RGB))
axes[0, 1].set_title(f'Classe A — Ruído Gaussiano\n(μ={MEDIA}, σ={DESVIO_PADRAO})', fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

axes[1, 0].imshow(cv2.cvtColor(img_B, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title('Classe B — Original', fontsize=12, fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(cv2.cvtColor(img_B_gauss, cv2.COLOR_BGR2RGB))
axes[1, 1].set_title(f'Classe B — Ruído Gaussiano\n(μ={MEDIA}, σ={DESVIO_PADRAO})', fontsize=12, fontweight='bold')
axes[1, 1].axis('off')

plt.suptitle('Comparação: Original vs Ruído Gaussiano', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Análise do Ruído Gaussiano

**Parâmetros utilizados:** média (μ) = 0, desvio padrão (σ) = 30

**Impacto visual observado:** O ruído gaussiano adiciona uma granulação uniforme por toda a imagem, como uma "névoa" que reduz a nitidez e suaviza detalhes finos. A imagem parece "embaçada" e com textura arenosa.

**O ruído está distribuído uniformemente na imagem?**
Sim. O ruído gaussiano, por definição, é adicionado a todos os pixels da imagem de forma estatisticamente uniforme, seguindo uma distribuição normal. Cada pixel recebe uma perturbação aleatória com a mesma probabilidade, o que resulta em um ruído visualmente homogêneo em toda a imagem.

**Que tipo de problema esse ruído pode causar para algoritmos de visão computacional?**
- **Detecção de bordas:** bordas ficam menos definidas, gerando falsos positivos ou falhas na detecção
- **Classificação:** texturas e padrões são corrompidos, podendo confundir classificadores
- **Segmentação:** limites entre regiões ficam borrados, prejudicando a separação de objetos
- **Extração de features:** descritores como SIFT/SURF podem gerar keypoints espúrios

In [ ]:
# Parâmetros do ruído sal e pimenta
PROB_SP = 0.05  # 5% dos pixels afetados

# Aplicar ruído sal e pimenta nas duas imagens
img_A_sp = adicionar_ruido_sal_pimenta(img_A, probabilidade=PROB_SP)
img_B_sp = adicionar_ruido_sal_pimenta(img_B, probabilidade=PROB_SP)

total_pixels_A = img_A.shape[0] * img_A.shape[1]
total_pixels_B = img_B.shape[0] * img_B.shape[1]
print(f"Parâmetros do Ruído Sal e Pimenta: probabilidade = {PROB_SP*100:.0f}%")
print(f"Classe A: ~{int(total_pixels_A * PROB_SP)} pixels alterados de {total_pixels_A} totais")
print(f"Classe B: ~{int(total_pixels_B * PROB_SP)} pixels alterados de {total_pixels_B} totais")

# Exibir comparação: Original vs Ruído Sal e Pimenta
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

axes[0, 0].imshow(cv2.cvtColor(img_A, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title('Classe A — Original', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(cv2.cvtColor(img_A_sp, cv2.COLOR_BGR2RGB))
axes[0, 1].set_title(f'Classe A — Ruído Sal e Pimenta\n({PROB_SP*100:.0f}% dos pixels)', fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

axes[1, 0].imshow(cv2.cvtColor(img_B, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title('Classe B — Original', fontsize=12, fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(cv2.cvtColor(img_B_sp, cv2.COLOR_BGR2RGB))
axes[1, 1].set_title(f'Classe B — Ruído Sal e Pimenta\n({PROB_SP*100:.0f}% dos pixels)', fontsize=12, fontweight='bold')
axes[1, 1].axis('off')

plt.suptitle('Comparação: Original vs Ruído Sal e Pimenta', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Análise do Ruído Sal e Pimenta

**Porcentagem de pixels alterados:** 5% (2.5% sal + 2.5% pimenta)

**Como esse ruído se diferencia visualmente do ruído gaussiano?**
O ruído sal e pimenta é **impulsivo** — ele substitui pixels individuais por valores extremos (branco puro = 255 ou preto puro = 0), criando pontos isolados e bem visíveis na imagem. Já o ruído gaussiano altera **todos** os pixels de forma sutil e contínua, gerando uma granulação difusa. O sal e pimenta é mais "pontual", enquanto o gaussiano é mais "espalhado".

**Em quais situações esse ruído aparece em sistemas reais de captura de imagem?**
- **Sensores defeituosos:** pixels mortos (stuck pixels) em câmeras digitais
- **Erros de transmissão:** perda de bits durante transferência de dados de imagem
- **Memória corrompida:** falhas no armazenamento que alteram valores de pixels aleatórios
- **Interferência eletromagnética:** em sistemas de captura industrial e câmeras de vigilância
- **Digitalização de documentos:** scanners com defeitos pontuais

---
## Parte 3 — Aplicação de Filtro Passa-Baixa Gaussiano (Suavização)

Aplicação de filtros gaussianos de suavização nas imagens com ruído para tentar reduzir o ruído e recuperar a qualidade visual. Testando dois tamanhos de kernel:
- **Kernel 3×3** — suavização leve
- **Kernel 7×7** — suavização mais intensa

In [ ]:
# Aplicar filtro gaussiano nas imagens com RUÍDO GAUSSIANO
# Kernel 3x3
img_A_gauss_k3 = cv2.GaussianBlur(img_A_gauss, (3, 3), 0)
img_B_gauss_k3 = cv2.GaussianBlur(img_B_gauss, (3, 3), 0)

# Kernel 7x7
img_A_gauss_k7 = cv2.GaussianBlur(img_A_gauss, (7, 7), 0)
img_B_gauss_k7 = cv2.GaussianBlur(img_B_gauss, (7, 7), 0)

# Exibir resultados — Classe A
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(cv2.cvtColor(img_A, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original', fontsize=11, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(img_A_gauss, cv2.COLOR_BGR2RGB))
axes[1].set_title('Com Ruído Gaussiano', fontsize=11, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(cv2.cvtColor(img_A_gauss_k3, cv2.COLOR_BGR2RGB))
axes[2].set_title('Filtro Gaussiano 3×3', fontsize=11, fontweight='bold')
axes[2].axis('off')

axes[3].imshow(cv2.cvtColor(img_A_gauss_k7, cv2.COLOR_BGR2RGB))
axes[3].set_title('Filtro Gaussiano 7×7', fontsize=11, fontweight='bold')
axes[3].axis('off')

plt.suptitle('Classe A — Filtragem Passa-Baixa no Ruído Gaussiano', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Exibir resultados — Classe B
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(cv2.cvtColor(img_B, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original', fontsize=11, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(img_B_gauss, cv2.COLOR_BGR2RGB))
axes[1].set_title('Com Ruído Gaussiano', fontsize=11, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(cv2.cvtColor(img_B_gauss_k3, cv2.COLOR_BGR2RGB))
axes[2].set_title('Filtro Gaussiano 3×3', fontsize=11, fontweight='bold')
axes[2].axis('off')

axes[3].imshow(cv2.cvtColor(img_B_gauss_k7, cv2.COLOR_BGR2RGB))
axes[3].set_title('Filtro Gaussiano 7×7', fontsize=11, fontweight='bold')
axes[3].axis('off')

plt.suptitle('Classe B — Filtragem Passa-Baixa no Ruído Gaussiano', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Aplicar filtro gaussiano nas imagens com RUÍDO SAL E PIMENTA
# Kernel 3x3
img_A_sp_k3 = cv2.GaussianBlur(img_A_sp, (3, 3), 0)
img_B_sp_k3 = cv2.GaussianBlur(img_B_sp, (3, 3), 0)

# Kernel 7x7
img_A_sp_k7 = cv2.GaussianBlur(img_A_sp, (7, 7), 0)
img_B_sp_k7 = cv2.GaussianBlur(img_B_sp, (7, 7), 0)

# Exibir resultados — Classe A
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(cv2.cvtColor(img_A, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original', fontsize=11, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(img_A_sp, cv2.COLOR_BGR2RGB))
axes[1].set_title('Com Ruído Sal e Pimenta', fontsize=11, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(cv2.cvtColor(img_A_sp_k3, cv2.COLOR_BGR2RGB))
axes[2].set_title('Filtro Gaussiano 3×3', fontsize=11, fontweight='bold')
axes[2].axis('off')

axes[3].imshow(cv2.cvtColor(img_A_sp_k7, cv2.COLOR_BGR2RGB))
axes[3].set_title('Filtro Gaussiano 7×7', fontsize=11, fontweight='bold')
axes[3].axis('off')

plt.suptitle('Classe A — Filtragem Passa-Baixa no Ruído Sal e Pimenta', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Exibir resultados — Classe B
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(cv2.cvtColor(img_B, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original', fontsize=11, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(img_B_sp, cv2.COLOR_BGR2RGB))
axes[1].set_title('Com Ruído Sal e Pimenta', fontsize=11, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(cv2.cvtColor(img_B_sp_k3, cv2.COLOR_BGR2RGB))
axes[2].set_title('Filtro Gaussiano 3×3', fontsize=11, fontweight='bold')
axes[2].axis('off')

axes[3].imshow(cv2.cvtColor(img_B_sp_k7, cv2.COLOR_BGR2RGB))
axes[3].set_title('Filtro Gaussiano 7×7', fontsize=11, fontweight='bold')
axes[3].axis('off')

plt.suptitle('Classe B — Filtragem Passa-Baixa no Ruído Sal e Pimenta', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Análise dos Filtros Passa-Baixa

**O filtro conseguiu reduzir o ruído?**
Sim. O filtro gaussiano conseguiu reduzir significativamente o ruído em ambos os casos. No ruído gaussiano, a suavização foi bastante eficaz, pois o filtro gaussiano atua na mesma "frequência" do ruído. No ruído sal e pimenta, o filtro reduziu o impacto visual, mas não eliminou completamente os pontos — para esse tipo de ruído, um filtro de mediana seria mais eficaz.

**Que detalhes da imagem foram perdidos?**
- Bordas ficaram mais suaves e menos definidas
- Texturas finas e padrões de alta frequência foram atenuados
- Pequenos detalhes e contornos delicados foram borrados
- Quanto maior o kernel, maior a perda de detalhes

**Qual kernel apresentou melhor equilíbrio entre suavização e preservação de detalhes?**
O **kernel 3×3** apresentou o melhor equilíbrio. Ele conseguiu reduzir uma quantidade perceptível de ruído enquanto preservou melhor as bordas e detalhes importantes da imagem. O kernel 7×7, embora mais eficaz na remoção de ruído, causou um borramento excessivo que comprometeu a qualidade visual e a utilidade da imagem para tarefas de visão computacional.

---
## Parte 4 — Aplicação de Filtro Passa-Alta (Realce de Bordas)

Aplicação de filtros passa-alta para realçar bordas e detalhes das imagens originais, utilizando:
- **Filtro Laplaciano** — detecta mudanças bruscas de intensidade em todas as direções
- **Kernel High-Pass clássico** — realça bordas subtraindo a versão suavizada

In [ ]:
# Converter para escala de cinza para melhor visualização das bordas
img_A_gray = cv2.cvtColor(img_A, cv2.COLOR_BGR2GRAY)
img_B_gray = cv2.cvtColor(img_B, cv2.COLOR_BGR2GRAY)

# 1. Filtro Laplaciano
laplaciano_A = cv2.Laplacian(img_A_gray, cv2.CV_64F)
laplaciano_A = np.uint8(np.absolute(laplaciano_A))

laplaciano_B = cv2.Laplacian(img_B_gray, cv2.CV_64F)
laplaciano_B = np.uint8(np.absolute(laplaciano_B))

# 2. Kernel High-Pass clássico (3x3)
kernel_hp = np.array([[-1, -1, -1],
                      [-1,  8, -1],
                      [-1, -1, -1]])

highpass_A = cv2.filter2D(img_A_gray, cv2.CV_64F, kernel_hp)
highpass_A = np.uint8(np.absolute(highpass_A))

highpass_B = cv2.filter2D(img_B_gray, cv2.CV_64F, kernel_hp)
highpass_B = np.uint8(np.absolute(highpass_B))

# Exibir resultados — Classe A
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(img_A_gray, cmap='gray')
axes[0].set_title('Classe A — Original (Escala de Cinza)', fontsize=11, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(laplaciano_A, cmap='gray')
axes[1].set_title('Classe A — Filtro Laplaciano', fontsize=11, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(highpass_A, cmap='gray')
axes[2].set_title('Classe A — High-Pass Kernel', fontsize=11, fontweight='bold')
axes[2].axis('off')

plt.suptitle('Classe A — Filtros Passa-Alta', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Exibir resultados — Classe B
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(img_B_gray, cmap='gray')
axes[0].set_title('Classe B — Original (Escala de Cinza)', fontsize=11, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(laplaciano_B, cmap='gray')
axes[1].set_title('Classe B — Filtro Laplaciano', fontsize=11, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(highpass_B, cmap='gray')
axes[2].set_title('Classe B — High-Pass Kernel', fontsize=11, fontweight='bold')
axes[2].axis('off')

plt.suptitle('Classe B — Filtros Passa-Alta', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Análise dos Filtros Passa-Alta

**Que estruturas da imagem foram destacadas?**
Os filtros passa-alta destacaram as **bordas**, **contornos** e **transições abruptas de intensidade** nas imagens. Regiões com mudanças bruscas de cor ou luminosidade — como limites entre objetos, texturas detalhadas e padrões de alta frequência — ficaram evidenciados em branco contra um fundo escuro.

**Bordas ficaram mais visíveis?**
Sim, significativamente. Tanto o filtro Laplaciano quanto o kernel high-pass realçaram claramente as bordas presentes nas imagens. O Laplaciano detectou bordas em todas as direções de forma isotrópica, enquanto o kernel high-pass clássico (com peso central 8) produziu um realce um pouco mais intenso, destacando contornos com maior contraste.

**Esse tipo de filtro ajudaria em quais tarefas de visão computacional?**
- **Detecção de objetos:** identificação de contornos para localizar objetos na cena
- **Segmentação de imagens:** separação de regiões com base nas bordas detectadas
- **Reconhecimento de padrões:** extração de features baseadas em bordas (edge-based features)
- **OCR (Reconhecimento de caracteres):** realce dos contornos de letras e números
- **Inspeção industrial:** detecção de defeitos, rachaduras e irregularidades em superfícies
- **Navegação autônoma:** identificação de faixas de trânsito e obstáculos

---
## Parte 5 — Comparação Final

Tabela comparativa dos resultados obtidos em todas as etapas do processamento, com análise consolidada do impacto dos ruídos e da eficácia dos filtros aplicados.

In [ ]:
# Painel de comparação completo — Classe A
fig, axes = plt.subplots(2, 5, figsize=(25, 10))

titulos = ['Original', 'Ruído Gaussiano', 'Ruído Sal e Pimenta',
           'Filtrada (Gauss 3×3)', 'Passa-Alta (Laplaciano)']

# Linha 1 — Classe A
imgs_A = [img_A, img_A_gauss, img_A_sp, img_A_gauss_k3, None]
for i, (img, titulo) in enumerate(zip(imgs_A, titulos)):
    if i == 4:
        axes[0, i].imshow(laplaciano_A, cmap='gray')
    else:
        axes[0, i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0, i].set_title(titulo, fontsize=11, fontweight='bold')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Classe A', fontsize=13, fontweight='bold')

# Linha 2 — Classe B
imgs_B = [img_B, img_B_gauss, img_B_sp, img_B_gauss_k3, None]
for i, (img, titulo) in enumerate(zip(imgs_B, titulos)):
    if i == 4:
        axes[1, i].imshow(laplaciano_B, cmap='gray')
    else:
        axes[1, i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[1, i].set_title(titulo, fontsize=11, fontweight='bold')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('Classe B', fontsize=13, fontweight='bold')

plt.suptitle('Comparação Final — Todas as Etapas do Processamento',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Tabela de Comparação

| Etapa | Resultado Observado |
|---|---|
| **Imagem original** | Imagem nítida, com detalhes preservados e sem artefatos visíveis. Serve como referência (ground truth) para todas as comparações. |
| **Imagem com ruído gaussiano** | Granulação uniforme em toda a imagem, como uma "névoa" que reduz nitidez. Texturas finas ficam comprometidas, mas a estrutura geral da imagem ainda é reconhecível. |
| **Imagem com ruído sal e pimenta** | Pontos brancos e pretos isolados distribuídos aleatoriamente. Menos degradação global que o gaussiano, mas os pontos são muito visíveis e podem confundir algoritmos de detecção de features. |
| **Imagem filtrada com gaussiano (3×3)** | Redução perceptível do ruído com boa preservação de bordas e detalhes. Suavização leve que melhora a qualidade visual sem borrar excessivamente. |
| **Imagem com filtro passa-alta** | Bordas e contornos claramente realçados. Regiões homogêneas aparecem escuras, enquanto transições de intensidade ficam destacadas em branco. |

### Respostas da Comparação Final

**Qual ruído degradou mais a imagem?**
O **ruído gaussiano** (com σ=30) degradou mais a imagem de forma global, pois afeta todos os pixels simultaneamente, reduzindo a nitidez geral e corrompendo texturas em toda a extensão da imagem. O ruído sal e pimenta, embora visualmente impactante, afeta apenas uma pequena fração dos pixels (5%) e preserva a informação original nos demais.

**O filtro gaussiano conseguiu recuperar a qualidade visual?**
Parcialmente. O filtro gaussiano foi eficaz na redução do ruído gaussiano, pois ambos operam na mesma faixa de frequência. A imagem filtrada com kernel 3×3 apresentou boa recuperação visual. Porém, a recuperação nunca é perfeita — há sempre um trade-off entre remoção de ruído e perda de detalhes. Para o ruído sal e pimenta, o filtro gaussiano foi menos eficaz; um filtro de mediana (`cv2.medianBlur`) seria mais indicado para esse tipo de ruído impulsivo.

**Em quais aplicações de visão computacional seria importante aplicar filtros antes do processamento?**
- **Detecção de objetos (YOLO, SSD):** imagens ruidosas geram detecções falsas; pré-filtragem melhora a precisão
- **Reconhecimento facial:** ruído compromete a extração de landmarks faciais
- **Veículos autônomos:** câmeras capturam ruído em condições de baixa luz; filtros são essenciais para segurança
- **Imagens médicas (raio-X, tomografia):** ruído pode mascarar diagnósticos; filtragem é etapa obrigatória
- **OCR em documentos digitalizados:** ruído de scanner prejudica o reconhecimento de caracteres
- **Inspeção industrial:** defeitos podem ser confundidos com ruído se não houver pré-processamento adequado

---
## Conclusão

Nesta atividade, exploramos na prática os conceitos fundamentais de **filtragem de imagens** e **análise de ruído** em visão computacional:

1. **Ruídos** — O ruído gaussiano e o sal e pimenta degradam as imagens de formas distintas: o primeiro de maneira global e contínua, o segundo de forma pontual e impulsiva.

2. **Filtro Passa-Baixa (Gaussiano)** — É eficaz para suavizar e reduzir ruído, especialmente o gaussiano, mas sempre há um compromisso entre remoção de ruído e preservação de detalhes. O kernel 3×3 ofereceu o melhor equilíbrio.

3. **Filtro Passa-Alta (Laplaciano)** — Realça bordas e detalhes de alta frequência, sendo fundamental para tarefas de detecção de contornos e segmentação.

4. **Importância prática** — Em pipelines reais de visão computacional, a etapa de pré-processamento com filtros é essencial para garantir que os algoritmos subsequentes (classificação, detecção, segmentação) operem sobre dados de qualidade adequada.